In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False


def load_dataset(filename, cols=None):
    BASE_DIR = './data'
    EXTENSION = '.csv'
    TARGET_PATH = f'{BASE_DIR}/{filename}{EXTENSION}'
    try:
        return pd.read_csv(TARGET_PATH)
    except FileNotFoundError:
        return None
    except UnicodeDecodeError:
        return pd.read_csv(TARGET_PATH, encoding='euc-kr', usecols=cols if cols else None)

In [2]:
d002 = load_dataset('물동량_추이')
d003 = load_dataset('수송통계')

In [3]:
d002.head(3)

,년도,항구분,총계,외항소계,외항입항,외항출항,외항입항환적,외항출항환적,내항연안화물
0,1993,북항,69468242,57526251,26152577,31373674,0,0,11941991
1,1994,북항,81680460,67960869,33397436,34563433,0,0,13719591
2,1994,감천,3429201,1740411,1662278,78133,0,0,1688790


In [4]:
d003.head(3)

,년도,합계,자성대부두,신선대부두,감만부두,우암부두,감천한진,신감만부두,일반부두,신항1부두,신항2부두,신항3부두,신항4부두,신항5부두,다목적부두등
0,1994,48140,0,0,0,0,0,0,48140,0,0,0,0,0,0
1,1995,59775,0,622,0,0,0,0,59153,0,0,0,0,0,0
2,1996,4809217,1673341,1302513,0,21171,0,0,1812192,0,0,0,0,0,0


In [5]:
import math

def rebuild_proposal_evaluation():
    d013_cols = ["항구명", "호출부호", "년도", "항차", "외내항구분", "선박종류", "입항목적", "계선장소시설코드", "계선장소명",
                 "전출항지출항일시", "신고일자", "입항일자", "허가일자", "출항예정일시", "정보생성일시"]
    reports = load_dataset("입항신고", cols=d013_cols).query("항구명 == '부산'")
    work_cols = [c for c in d013_cols if c not in ["허가일자", "정보생성일시"]]
    reports = reports.sort_values("정보생성일시").drop_duplicates(work_cols, keep="last").copy()
    year = pd.to_numeric(reports["입항일자"].str[:4], errors="coerce")
    reports["입항일자"] = pd.to_datetime(reports["입항일자"].where(year.between(2020, 2030)), errors="coerce")
    reports = reports[(reports["입항일자"] >= "2024-05-01") & (reports["입항일자"] < "2025-08-01")].copy()

    calls = reports.drop_duplicates(["호출부호", "년도", "항차"], keep="last")
    reports["선박입항수"] = reports["호출부호"].map(calls.groupby("호출부호").size())
    weekend = reports[reports["입항일자"].dt.dayofweek.isin([4, 5, 6])]
    regular_share = tuple((weekend["선박입항수"] >= k).mean() for k in [30, 15, 5])

    d012 = load_dataset("선박관제정보", cols=["관제항구", "호출부호", "신고입항년도", "신고입항횟수", "입항일시", "상태구분", "정보생성일시"])
    control_all = d012[(d012["관제항구"] == "부산") & (d012["상태구분"] != "삭제")].copy()
    call_key = ["호출부호", "신고입항년도", "신고입항횟수"]
    control = control_all.sort_values("정보생성일시").drop_duplicates(call_key, keep="last")
    linked = calls.merge(control[call_key], left_on=["호출부호", "년도", "항차"], right_on=call_key, how="left", indicator=True)
    unlinked = (linked["_merge"] == "left_only").sum()

    daily = pd.to_datetime(control_all["입항일시"]).dt.normalize().value_counts().sort_index().asfreq("D", fill_value=0)
    z_daily = (daily - daily.mean()) / daily.std()
    alert_days = daily.index[z_daily.abs() > 3]
    holiday_dates = pd.to_datetime(["2024-05-06", "2024-05-15", "2024-06-06", "2024-08-15", "2024-09-16", "2024-09-17",
                                    "2024-09-18", "2024-10-01", "2024-10-03", "2024-10-09", "2024-12-25", "2025-01-01",
                                    "2025-01-27", "2025-01-28", "2025-01-29", "2025-01-30", "2025-03-01", "2025-03-03",
                                    "2025-05-05", "2025-05-06", "2025-06-03", "2025-06-06"])
    explained = [day == daily.index.max() or (abs(holiday_dates - day) <= pd.Timedelta(days=1)).any() for day in alert_days]
    explained_rate = sum(explained) / len(alert_days)

    discount_rows = (load_dataset("공컨환적할인", cols=["신고청"])["신고청"] == "부산").sum()

    weekend_n = len(weekend) * 12 / 15
    unlinked_n = unlinked * 12 / 15
    alerts_n = len(alert_days) * 365 / len(daily)

    wage = (30_000, 35_000, 40_000)
    factors = {
        "A": [(weekend_n,) * 3, regular_share, (0.5, 0.7, 0.9), (5, 10, 15), wage],
        "B": [(unlinked_n,) * 3, (0.05, 0.1, 0.2), (3, 5, 10), wage],
        "C": [(discount_rows,) * 3, (0.6, 0.8, 0.95), (0.5, 1.0, 2.0), wage],
        "D": [(alerts_n,) * 3, (0.5, explained_rate, 0.75), (30, 60, 120), wage],
    }
    benefits = pd.DataFrame({name: [math.prod(f[i] for f in fs) / 60 for i in range(3)] for name, fs in factors.items()},
                            index=["보수", "기준", "낙관"]).T
    costs = pd.Series({"A": 80_000_000, "B": 20_000_000, "C": 50_000_000, "D": 5_000_000})

    scores = pd.DataFrame({"A": [4, 4, 3, 2, 3], "B": [3, 2, 2, 3, 3], "C": [5, 5, 4, 4, 4], "D": [5, 5, 5, 5, 5]},
                          index=["기술", "데이터", "조직/인력", "제도/규정", "비용/기간"]).T

    summary = pd.DataFrame({
        "요약": {"A": "주말 입항 정형 신고의 즉시 처리", "B": "내항 신고/관제 미연결 원인 규명과 연결",
                "C": "공컨테이너 환적 할인금액 자동 산정", "D": "입항 척수 경보에 공휴일, 데이터 경계 표시"},
        "근거강도": {"A": "A1 강, A2 약", "B": "B1 강, B2 약", "C": "단일 출처", "D": "D1 약"},
        "비화폐효과": {"A": "주말 입항분의 허가 확정 대기 단축", "B": "항만 교통 데이터의 연결성",
                   "C": "산정 기준의 일관성", "D": "경보 피로 감소"},
    })
    summary["편익_보수_만원"] = (benefits["보수"] / 10_000).round(0)
    summary["편익_기준_만원"] = (benefits["기준"] / 10_000).round(0)
    summary["편익_낙관_만원"] = (benefits["낙관"] / 10_000).round(0)
    summary["도입비용_만원"] = costs / 10_000
    summary["회수기간_년"] = (costs / benefits["기준"]).round(1)
    summary["실행가능성"] = scores.mean(axis=1).round(2)
    summary["실행병목"] = scores.idxmin(axis=1)
    summary.index.name = "개선안"
    return summary

In [6]:
rebuild_proposal_evaluation()

,요약,근거강도,비화폐효과,편익_보수_만원,편익_기준_만원,편익_낙관_만원,도입비용_만원,회수기간_년,실행가능성,실행병목
개선안,,,,,,,,,,
A,주말 입항 정형 신고의 즉시 처리,"A1 강, A2 약",주말 입항분의 허가 확정 대기 단축,1248.0,4928.0,13597.0,8000.0,1.6,3.2,제도·규정
B,내항 신고·관제 미연결 원인 규명과 연결,"B1 강, B2 약",항만 교통 데이터의 연결성,92.0,357.0,1634.0,2000.0,5.6,2.6,데이터
C,공컨테이너 환적 할인금액 자동 산정,단일 출처,산정 기준의 일관성,2620.0,8152.0,22126.0,5000.0,0.6,4.4,조직·인력
D,"입항 척수 경보에 공휴일, 데이터 경계 표시",D1 약,경보 피로 감소,5.0,14.0,38.0,500.0,35.8,5.0,기술
